# Basic ML Model Deployment

## Import libraries

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import KNNImputer,SimpleImputer
from sklearn.preprocessing import StandardScaler
import pickle

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor

## Fetch Data

In [2]:
data=pd.read_csv('https://raw.githubusercontent.com/tkseneee/Dataset/master/Loan_data_ver2.csv')
#data=pd.read_csv('Loan_data_ver2.csv')

## Explore Data

In [3]:
data.shape

(614, 6)

In [4]:
data.dtypes

Married             object
Education           object
ApplicantIncome      int64
LoanAmount         float64
Credit_History     float64
Loan_Status        float64
dtype: object

In [5]:
data.head(2)

,Married,Education,ApplicantIncome,LoanAmount,Credit_History,Loan_Status
0,No,Graduate,5849,NaN,1.0,0.10
1,Yes,Graduate,4583,128.0,1.0,0.32


In [6]:
# fetch features with missing values
data.isnull().sum()

Married             3
Education           0
ApplicantIncome     0
LoanAmount         22
Credit_History     50
Loan_Status         0
dtype: int64

3 features namely - Married,LoanAmount,Credit_History has missing values

In [7]:
data['Married'].value_counts()

Married
Yes    398
No     213
Name: count, dtype: int64

In [8]:
data['Education'].value_counts()

Education
Graduate        449
Not Graduate    127
HSC              38
Name: count, dtype: int64

In [9]:
# segreegating target & feature
X=data.drop('Loan_Status', axis=1)
y=data['Loan_Status']

In [13]:
# spliting data into train & validation set
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.3,random_state=48)

In [14]:
# fetching numeric features list
feat_num=list(X.select_dtypes(include=np.number).columns)


In [15]:
# fetching categorical features  list
feat_cat=list(X.select_dtypes(exclude=np.number).columns)

In [16]:
feat_cat

['Married', 'Education']

## Defining Data processing & Modeling  Pipeline

In [17]:
#  pipeline for numeric atures -missing values replacement using k-Nearest Neighbors follwed by StandardScaler() 
num_pipe=Pipeline([('imputer',KNNImputer()),('std_scale',StandardScaler())])



In [18]:
# pipeline for categorical faetures - missing category replacement by new category i.e. missing followed by one hot encoding 
feat_pipe = Pipeline([('imputer',SimpleImputer(strategy='constant', fill_value='Missing')), 
                      ('one_hot',(OneHotEncoder()))]) 



In [19]:
#combine data processing pipeline
data_pipeline=ColumnTransformer([('numeric',num_pipe,feat_num),
                                 ('categorical',feat_pipe, feat_cat)],
                                remainder='passthrough')



In [20]:
data_pipeline

ColumnTransformer(remainder='passthrough',
                  transformers=[('numeric',
                                 Pipeline(steps=[('imputer', KNNImputer()),
                                                 ('std_scale',
                                                  StandardScaler())]),
                                 ['ApplicantIncome', 'LoanAmount',
                                  'Credit_History']),
                                ('categorical',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(fill_value='Missing',
                                                                strategy='constant')),
                                                 ('one_hot', OneHotEncoder())]),
                                 ['Married', 'Education'])])

In [21]:
# adding ml-model into pipeline 
full_pipe=Pipeline([('pre_process',data_pipeline),('model',RandomForestRegressor())])

In [22]:
# training
full_pipe.fit(X_train,y_train)

Pipeline(steps=[('pre_process',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('numeric',
                                                  Pipeline(steps=[('imputer',
                                                                   KNNImputer()),
                                                                  ('std_scale',
                                                                   StandardScaler())]),
                                                  ['ApplicantIncome',
                                                   'LoanAmount',
                                                   'Credit_History']),
                                                 ('categorical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(fill_value='Missing',
                                                                                 strategy='constant')),
                                                                  ('one_hot',
                                                                   OneHotEncoder())]),
                                                  ['Married', 'Education'])])),
                ('model', RandomForestRegressor())])

In [23]:
# prediction
full_pipe.predict(X_test)

array([0.1379, 0.2212, 0.5585, 0.3931, 0.1575, 0.0945, 0.9539, 0.2036,
       0.2398, 0.2837, 0.1919, 0.9751, 0.3154, 0.1286, 0.0517, 0.1072,
       0.1254, 0.049 , 0.0476, 0.2645, 0.2992, 0.0798, 0.2739, 0.49  ,
       0.6368, 0.1663, 0.2657, 0.2372, 0.2476, 0.016 , 0.1215, 0.2025,
       0.4043, 0.9442, 0.1248, 0.2845, 0.1937, 0.6268, 0.9521, 0.1725,
       0.289 , 0.1541, 0.2989, 0.3244, 0.4043, 0.4872, 0.1363, 0.1721,
       0.2607, 0.1838, 0.064 , 0.4383, 0.0698, 0.1139, 0.9621, 0.0079,
       0.4153, 0.322 , 0.3017, 0.2157, 0.2848, 0.8364, 0.2783, 0.1596,
       0.4566, 0.2182, 0.0793, 0.2681, 0.2136, 0.2063, 0.3976, 0.1491,
       0.3014, 0.3043, 0.069 , 0.8229, 0.1109, 0.4606, 0.6374, 0.0427,
       0.2612, 0.1117, 0.2764, 0.7723, 0.462 , 0.1299, 0.1869, 0.0817,
       0.0585, 0.5105, 0.0625, 0.1885, 0.2362, 0.534 , 0.3631, 0.2378,
       0.966 , 0.0764, 0.0909, 0.387 , 0.4894, 0.0735, 0.0046, 0.051 ,
       0.0716, 0.0728, 0.3297, 0.2364, 0.0203, 0.1091, 0.1555, 0.1004,
      

In [24]:
## can store numeric and categorical variables also as pickle file
pickle.dump(feat_num,open('feat_numv1','wb'))
pickle.dump(feat_cat,open('feat_catv1','wb'))

 

## Store the model as pickle file 

In [25]:
pickle.dump(full_pipe,open('full_pipeline','wb'))